In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q diffusers transformers accelerate peft torchmetrics pandas matplotlib pillow safetensors huggingface_hub
!pip uninstall -y torchao

In [ ]:
import importlib.util

print("torchao instalado?", importlib.util.find_spec("torchao") is not None)

torchao instalado? False


In [ ]:
import importlib.util, shutil, os, glob, site

# Remove possíveis sobras em site-packages
possiveis_pastas = []

for pasta in site.getsitepackages():
    possiveis_pastas.extend(glob.glob(os.path.join(pasta, "torchao*")))

try:
    possiveis_pastas.extend(glob.glob(os.path.join(site.getusersitepackages(), "torchao*")))
except:
    pass

for caminho in possiveis_pastas:
    print("Removendo:", caminho)
    if os.path.isdir(caminho):
        shutil.rmtree(caminho, ignore_errors=True)
    else:
        try:
            os.remove(caminho)
        except:
            pass

print("torchao instalado?", importlib.util.find_spec("torchao") is not None)

torchao instalado? False


In [ ]:
import torch

print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: ative a GPU T4 no Colab.")

CUDA disponível: True
GPU: Tesla T4


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from pathlib import Path

MODEL_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# Seu LoRA publicado no Hugging Face
LORA_PATH = "pedrohsmoura/lora-brasilia-rank8"

# Pasta onde todos os resultados da Etapa 3 serão salvos
PASTA_AVALIACAO = Path("/content/drive/MyDrive/avaliacao_lora_brasilia")
PASTA_AVALIACAO.mkdir(parents=True, exist_ok=True)

print("Modelo base:", MODEL_BASE)
print("LoRA usado:", LORA_PATH)
print("Pasta de avaliação:", PASTA_AVALIACAO)

Modelo base: stable-diffusion-v1-5/stable-diffusion-v1-5
LoRA usado: pedrohsmoura/lora-brasilia-rank8
Pasta de avaliação: /content/drive/MyDrive/avaliacao_lora_brasilia


In [ ]:
prompts = [
    "estilo_brasilia, uma praça pública com arquitetura modernista de brasilia, concreto branco, curvas elegantes e ceu azul do cerrado",
    "estilo_brasilia, um museu futurista inspirado na arquitetura modernista de brasilia, colunas monumentais, concreto e vidro",
    "estilo_brasilia, uma avenida ampla com prédios públicos modernistas, pilotis, linhas limpas e céu azul intenso",
    "estilo_brasilia, um palácio governamental com curvas suaves, concreto branco, espelho d'água e paisagismo do cerrado",
    "estilo_brasilia, uma biblioteca pública modernista com rampas, colunas, concreto aparente e luz natural",
    "estilo_brasilia, uma estação cultural ao pôr do sol, arquitetura de brasilia, formas geométricas, concreto branco e céu do cerrado"
]

negative_prompt = "desfocado, deformado, baixa qualidade, texto, marca d'água, pessoas deformadas"

SEED_BASE = 42

In [ ]:
import torch, gc
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

assert device == "cuda", "Ative a GPU T4 antes de rodar esta etapa."

pasta_base = PASTA_AVALIACAO / "modelo_base"
pasta_base.mkdir(exist_ok=True)

pipe_base = StableDiffusionPipeline.from_pretrained(
    MODEL_BASE,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

pipe_base.enable_attention_slicing()

base_paths = []

for i, prompt in enumerate(prompts):
    generator = torch.Generator(device=device).manual_seed(SEED_BASE + i)

    imagem = pipe_base(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    caminho = pasta_base / f"base_prompt_{i+1:02d}.png"
    imagem.save(caminho)
    base_paths.append(caminho)

    print("Salvo:", caminho)

del pipe_base
gc.collect()
torch.cuda.empty_cache()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_01.png


...

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_04.png


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_05.png


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_06.png


In [ ]:
import torch, gc
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pasta_lora = PASTA_AVALIACAO / "modelo_lora"
pasta_lora.mkdir(exist_ok=True)

pipe_lora = StableDiffusionPipeline.from_pretrained(
    MODEL_BASE,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

pipe_lora.load_lora_weights(
    LORA_PATH,
    weight_name="pytorch_lora_weights.safetensors"
)

pipe_lora.enable_attention_slicing()

lora_paths = []

for i, prompt in enumerate(prompts):
    generator = torch.Generator(device=device).manual_seed(SEED_BASE + i)

    imagem = pipe_lora(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    caminho = pasta_lora / f"lora_prompt_{i+1:02d}.png"
    imagem.save(caminho)
    lora_paths.append(caminho)

    print("Salvo:", caminho)

del pipe_lora
gc.collect()
torch.cuda.empty_cache()

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_01.png


...

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_04.png


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_05.png


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_06.png


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(len(prompts), 2, figsize=(10, 30))

for i in range(len(prompts)):
    img_base = Image.open(base_paths[i])
    img_lora = Image.open(lora_paths[i])

    axes[i, 0].imshow(img_base)
    axes[i, 0].set_title(f"Modelo base — Prompt {i+1}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(img_lora)
    axes[i, 1].set_title(f"Modelo + LoRA — Prompt {i+1}")
    axes[i, 1].axis("off")

plt.tight_layout()

caminho_grade = PASTA_AVALIACAO / "grade_comparativa_base_vs_lora.png"
plt.savefig(caminho_grade, dpi=200, bbox_inches="tight")
plt.show()

print("Grade salva em:", caminho_grade)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pathlib import Path

PASTA_AVALIACAO = Path("/content/drive/MyDrive/avaliacao_lora_brasilia")

pasta_base = PASTA_AVALIACAO / "modelo_base"
pasta_lora = PASTA_AVALIACAO / "modelo_lora"

base_paths = [
    pasta_base / f"base_prompt_{i+1:02d}.png"
    for i in range(6)
]

lora_paths = [
    pasta_lora / f"lora_prompt_{i+1:02d}.png"
    for i in range(6)
]

print("Imagens base:")
for p in base_paths:
    print(p, p.exists())

print("\nImagens LoRA:")
for p in lora_paths:
    print(p, p.exists())

assert all(p.exists() for p in base_paths), "Alguma imagem do modelo base não foi encontrada."
assert all(p.exists() for p in lora_paths), "Alguma imagem do modelo LoRA não foi encontrada."

Imagens base:
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_01.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_02.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_03.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_04.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_05.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_base/base_prompt_06.png True

Imagens LoRA:
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_01.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_02.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_03.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_04.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora/lora_prompt_05.png True
/content/drive/MyDrive/avaliacao_lora_brasilia/modelo_lora

In [ ]:
import torch
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model_name = "openai/clip-vit-base-patch16"

clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

clip_model.eval()

def calcular_clipscore_manual(caminho_imagem, texto):
    imagem = Image.open(caminho_imagem).convert("RGB")

    inputs = clip_processor(
        text=[texto],
        images=imagem,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clip_model(**inputs)

        image_features = outputs.image_embeds
        text_features = outputs.text_embeds

        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        similaridade = torch.matmul(image_features, text_features.T).item()

    # Escala parecida com CLIPScore: similaridade cosseno x 100
    score = max(similaridade * 100, 0)

    return score


resultados_clip = []

for i, prompt in enumerate(prompts):
    score_base = calcular_clipscore_manual(base_paths[i], prompt)

    resultados_clip.append({
        "prompt_id": i + 1,
        "modelo": "base",
        "prompt": prompt,
        "clipscore": score_base
    })

    score_lora = calcular_clipscore_manual(lora_paths[i], prompt)

    resultados_clip.append({
        "prompt_id": i + 1,
        "modelo": "lora",
        "prompt": prompt,
        "clipscore": score_lora
    })

df_clip = pd.DataFrame(resultados_clip)

display(df_clip)

media_clip = df_clip.groupby("modelo")["clipscore"].mean().reset_index()

display(media_clip)

df_clip.to_csv(PASTA_AVALIACAO / "clipscore_resultados.csv", index=False)
media_clip.to_csv(PASTA_AVALIACAO / "clipscore_media_por_modelo.csv", index=False)

print("Resultados salvos em:", PASTA_AVALIACAO)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

,prompt_id,modelo,prompt,clipscore
0,1,base,"estilo_brasilia, uma praça pública com arquite...",29.840946
1,1,lora,"estilo_brasilia, uma praça pública com arquite...",28.508916
2,2,base,"estilo_brasilia, um museu futurista inspirado ...",31.908500
3,2,lora,"estilo_brasilia, um museu futurista inspirado ...",34.940815
4,3,base,"estilo_brasilia, uma avenida ampla com prédios...",33.303431
5,3,lora,"estilo_brasilia, uma avenida ampla com prédios...",32.163367
6,4,base,"estilo_brasilia, um palácio governamental com ...",29.447788
7,4,lora,"estilo_brasilia, um palácio governamental com ...",28.576821
8,5,base,"estilo_brasilia, uma biblioteca pública modern...",33.022922
9,5,lora,"estilo_brasilia, uma biblioteca pública modern...",33.609539


,modelo,clipscore
0,base,29.947087
1,lora,30.894576


Resultados salvos em: /content/drive/MyDrive/avaliacao_lora_brasilia


In [ ]:
from pathlib import Path

PROJETO = Path("/content/drive/MyDrive/atelie-generativo-brasilia-arquitetura")
ARQUIVO_LEGENDAS = PROJETO / "dados" / "legendas.txt"

print("Arquivo de legendas existe?", ARQUIVO_LEGENDAS.exists())

Arquivo de legendas existe? True


In [ ]:
if ARQUIVO_LEGENDAS.exists():
    with open(ARQUIVO_LEGENDAS, "r", encoding="utf-8") as f:
        legendas = [linha.strip() for linha in f.readlines() if linha.strip()]

    prompts_memorizacao = legendas[:10]

    prompts_memorizacao = [
        p if p.lower().startswith("estilo_brasilia") else "estilo_brasilia, " + p
        for p in prompts_memorizacao
    ]

else:
    prompts_memorizacao = [
        "estilo_brasilia, edifício modernista em concreto branco com colunas monumentais e céu azul do cerrado",
        "estilo_brasilia, fachada de arquitetura modernista de brasilia com linhas retas, concreto e vidro",
        "estilo_brasilia, prédio público com curvas elegantes, espelho d'água e paisagismo do cerrado",
        "estilo_brasilia, monumento urbano de brasilia com formas geométricas e concreto aparente",
        "estilo_brasilia, palácio modernista com pilares brancos, rampa e céu aberto",
        "estilo_brasilia, vista frontal de edifício institucional em concreto branco e linhas limpas",
        "estilo_brasilia, praça ampla com arquitetura monumental, gramado e céu azul",
        "estilo_brasilia, construção modernista com arcos, concreto claro e luz natural",
        "estilo_brasilia, cena urbana de brasilia com prédio público, curvas suaves e horizonte limpo",
        "estilo_brasilia, composição arquitetônica modernista com colunas, vidro, concreto branco e céu do cerrado"
    ]

for i, p in enumerate(prompts_memorizacao):
    print(i + 1, p)

1 estilo_brasilia, id_imagem_001.jpg|estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto
2 estilo_brasilia, id_imagem_002.jpg|estilo_arquitetura_brasilia, vista do topo do Teatro Nacional de Brasília Claúdio Santoro
3 estilo_brasilia, id_imagem_003.jpg|estilo_arquitetura_brasilia, vista da entrada do Teatro Nacional de Brasília Claúdio Santoro mostrando a rampa de acesso e a porta do teatro
4 estilo_brasilia, id_imagem_004.jpg|estilo_arquitetura_brasilia, vista lateral esquerda do Teatro Nacional de Brasília Claúdio Santoro mostrando parte da rodoviária do plano piloto
5 estilo_brasilia, id_imagem_005.jpg|estilo_arquitetura_brasilia, a imagem mostra o Memorial JK projetado pelo renomado arquiteto Oscar Niemeyer a composição destaca-se pelo forte contraste entre as estruturas  de concreto no topo deste pedestal encontra-se a estátua de bronze do ex-pre

In [ ]:
import torch, gc
from diffusers import StableDiffusionPipeline

MODEL_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_PATH = "pedrohsmoura/lora-brasilia-rank8"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pasta_mem = PASTA_AVALIACAO / "memorizacao_overfitting"
pasta_mem.mkdir(exist_ok=True)

pipe_lora = StableDiffusionPipeline.from_pretrained(
    MODEL_BASE,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

pipe_lora.load_lora_weights(
    LORA_PATH,
    weight_name="pytorch_lora_weights.safetensors"
)

pipe_lora.enable_attention_slicing()

mem_paths = []

for i, prompt in enumerate(prompts_memorizacao):
    generator = torch.Generator(device=device).manual_seed(1000 + i)

    imagem = pipe_lora(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    caminho = pasta_mem / f"memorizacao_{i+1:02d}.png"
    imagem.save(caminho)
    mem_paths.append(caminho)

    print("Salvo:", caminho)

del pipe_lora
gc.collect()
torch.cuda.empty_cache()

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/memorizacao_overfitting/memorizacao_01.png


...

  0%|          | 0/30 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['presidente brasileiro no qual o casal é retratado em uma pose afetuosa juscelino aparece de terno com o braço sobre os ombros de sarah enquanto ambos olham para frente com expressões serenas ao fundo']


Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/memorizacao_overfitting/memorizacao_09.png


  0%|          | 0/30 [00:00<?, ?it/s]

Salvo: /content/drive/MyDrive/avaliacao_lora_brasilia/memorizacao_overfitting/memorizacao_10.png


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(5, 2, figsize=(10, 25))
axes = axes.flatten()

for i, caminho in enumerate(mem_paths):
    img = Image.open(caminho)

    axes[i].imshow(img)
    axes[i].set_title(f"Prompt próximo ao treino {i+1}")
    axes[i].axis("off")

plt.tight_layout()

caminho_mem_grade = PASTA_AVALIACAO / "grade_memorizacao_overfitting.png"
plt.savefig(caminho_mem_grade, dpi=200, bbox_inches="tight")
plt.show()

print("Grade de memorização salva em:", caminho_mem_grade)

Output hidden; open in https://colab.research.google.com to view.